# Lab Assignment 6

In [1]:
import numpy as np

def sigmoid(z): return 1 / (1 + np.exp(-z))
def d_sigmoid(z): s = sigmoid(z); return s * (1 - s)

def relu(z): return np.maximum(0, z)
def d_relu(z): return (z > 0).astype(float)

def tanh(z): return np.tanh(z)
def d_tanh(z): return 1 - np.tanh(z)**2

def leaky_relu(z, alpha=0.01): return np.where(z > 0, z, alpha * z)
def d_leaky_relu(z, alpha=0.01): return np.where(z > 0, 1, alpha)

def softplus(z): return np.log(1 + np.exp(z))
def d_softplus(z): return 1 / (1 + np.exp(-z))

In [4]:
class DeepNetwork:
    def __init__(self, layer_dims, activation_type='relu'):
        self.params = {}
        self.L = len(layer_dims) - 1
        self.activation = activation_type

        for l in range(1, self.L + 1):
            self.params[f'W{l}'] = np.random.uniform(-0.5, 0.5, (layer_dims[l], layer_dims[l-1]))
            self.params[f'b{l}'] = np.zeros((layer_dims[l], 1))

    def forward(self, X):
        cache = {'A0': X.T}
        for l in range(1, self.L):
            Z = np.dot(self.params[f'W{l}'], cache[f'A{l-1}']) + self.params[f'b{l}']
            cache[f'Z{l}'] = Z
            if self.activation == 'relu': cache[f'A{l}'] = relu(Z)
            else: cache[f'A{l}'] = sigmoid(Z)

        ZL = np.dot(self.params[f'W{self.L}'], cache[f'A{self.L-1}']) + self.params[f'b{self.L}']
        cache[f'Z{self.L}'] = ZL
        cache[f'A{self.L}'] = ZL
        return cache[f'A{self.L}'], cache

    def backward(self, cache, Y):
        grads = {}
        m = Y.shape[0]
        dZ = (2/m) * (cache[f'A{self.L}'] - Y.T)

        for l in reversed(range(1, self.L + 1)):
            grads[f'dW{l}'] = np.dot(dZ, cache[f'A{l-1}'].T)
            grads[f'db{l}'] = np.sum(dZ, axis=1, keepdims=True)

            if l > 1:
                dA_prev = np.dot(self.params[f'W{l}'].T, dZ)
                if self.activation == 'relu':
                    dZ = dA_prev * d_relu(cache[f'Z{l-1}'])
                else:
                    dZ = dA_prev * d_sigmoid(cache[f'Z{l-1}'])
        return grads

    def update(self, grads, lr=0.01):
        for l in range(1, self.L + 1):
            self.params[f'W{l}'] -= lr * grads[f'dW{l}']
            self.params[f'b{l}'] -= lr * grads[f'db{l}']

In [5]:
np.random.seed(42)
X = np.random.uniform(-2, 2, (400, 3))
y = (np.sin(X[:,0]) + 0.5*(X[:,1]**2) - 0.8*X[:,2]).reshape(-1, 1)

architectures = {
    "Model A (Shallow)": [3, 4, 1],
    "Model B (Medium)": [3, 6, 6, 1],
    "Model C (Deep)": [3, 8, 8, 8, 8, 1],
    "Model D (ReLU)": [3, 8, 8, 8, 8, 8, 8, 8, 8, 1],
    "Model D (Sigmoid)": [3, 8, 8, 8, 8, 8, 8, 8, 8, 1]
}

print(f"{'Model':<20} | {'Final Loss':<12} | {'Grad Norm L1':<15} | {'Grad Norm Last'}")
print("-" * 75)

for name, dims in architectures.items():
    act = 'sigmoid' if "Sigmoid" in name else 'relu'
    nn = DeepNetwork(dims, act)

    for epoch in range(1001):
        AL, cache = nn.forward(X)
        grads = nn.backward(cache, y)
        nn.update(grads)

        if epoch == 1000:
            gn1 = np.sqrt(np.sum(np.square(grads['dW1'])))

            gnL = np.sqrt(np.sum(np.square(grads[f'dW{nn.L-1}'])))
            loss = np.mean((AL.T - y)**2)

            print(f"{name:<20} | {loss:<12.6f} | {gn1:<15.6f} | {gnL:.6f}")

Model                | Final Loss   | Grad Norm L1    | Grad Norm Last
---------------------------------------------------------------------------
Model A (Shallow)    | 0.111479     | 0.044987        | 0.044987
Model B (Medium)     | 0.072834     | 0.036565        | 0.021443
Model C (Deep)       | 0.030423     | 0.023639        | 0.015954
Model D (ReLU)       | 0.052776     | 0.417032        | 0.609742
Model D (Sigmoid)    | 1.743850     | 0.000006        | 0.000006
